In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

_DATA_CANDIDATES = lambda base: (base / "data", base / "Proyecto_Integrador" / "data")
DATA_DIR = next(
    (d for base in [Path.cwd(), *Path.cwd().parents]
     for d in _DATA_CANDIDATES(base)
     if (d / "bureau_balance.parquet").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("No se encontró data/bureau_balance.parquet en el proyecto")

bb = pd.read_parquet(DATA_DIR / "bureau_balance.parquet")

print(bb.shape)
print(bb.dtypes)
print(bb.head(10))
print(bb.isnull().sum())

(27299925, 3)
SK_ID_BUREAU      int64
MONTHS_BALANCE    int64
STATUS              str
dtype: object
   SK_ID_BUREAU  MONTHS_BALANCE STATUS
0       5715448               0      C
1       5715448              -1      C
2       5715448              -2      C
3       5715448              -3      C
4       5715448              -4      C
5       5715448              -5      C
6       5715448              -6      C
7       5715448              -7      C
8       5715448              -8      C
9       5715448              -9      0
SK_ID_BUREAU      0
MONTHS_BALANCE    0
STATUS            0
dtype: int64


In [3]:
status_counts = bb['STATUS'].value_counts()
meanings = {
    'C': 'Closed (cerrado)',
    'X': 'Unknown (sin información)',
    '0': 'Sin mora (al día)',
    '1': '1-29 días de mora',
    '2': '30-59 días de mora',
    '3': '60-89 días de mora',
    '4': '90-119 días de mora',
    '5': '120+ días de mora',
}
for s, cnt in status_counts.items():
    print(f"{s} = {meanings[s]}  →  {cnt:,}  ({cnt/len(bb)*100:.1f}%)")

C = Closed (cerrado)  →  13,646,993  (50.0%)
0 = Sin mora (al día)  →  7,499,507  (27.5%)
X = Unknown (sin información)  →  5,810,482  (21.3%)
1 = 1-29 días de mora  →  242,347  (0.9%)
5 = 120+ días de mora  →  62,406  (0.2%)
2 = 30-59 días de mora  →  23,419  (0.1%)
3 = 60-89 días de mora  →  8,924  (0.0%)
4 = 90-119 días de mora  →  5,847  (0.0%)


In [4]:
# ¿X aparece más en meses viejos?
bb['tramo'] = pd.cut(bb['MONTHS_BALANCE'],
                     bins=[-97,-72,-48,-24,-12,-1,0],
                     labels=['>6 años','4-6 años','2-4 años','1-2 años','<1 año','mes 0'])

print(bb.groupby('tramo', observed=True)['STATUS']
        .apply(lambda x: (x=='X').mean())
        .round(3))

# Créditos donde todos los meses son X
total_por_credito = bb.groupby('SK_ID_BUREAU').size().rename('total')
x_por_credito     = bb[bb['STATUS']=='X'].groupby('SK_ID_BUREAU').size().rename('x_count')
comparacion = pd.concat([total_por_credito, x_por_credito], axis=1).fillna(0)
todos_x = (comparacion['x_count'] == comparacion['total']).sum()
print(f"\nCréditos donde 100% meses son X: {todos_x:,} ({todos_x/len(total_por_credito)*100:.1f}%)")

tramo
>6 años     0.293
4-6 años    0.228
2-4 años    0.209
1-2 años    0.200
<1 año      0.191
mes 0       0.214
Name: STATUS, dtype: float64

Créditos donde 100% meses son X: 85,569 (10.5%)


In [5]:
mora_states = ['1','2','3','4','5']

# Primero crear columnas auxiliares — mucho más rápido que lambdas
bb['is_mora']  = bb['STATUS'].isin(mora_states).astype(int)
bb['is_0']     = (bb['STATUS'] == '0').astype(int)
bb['is_C']     = (bb['STATUS'] == 'C').astype(int)
bb['is_X']     = (bb['STATUS'] == 'X').astype(int)
bb['severity'] = bb['STATUS'].apply(lambda x: int(x) if x.isdigit() else 0)

# Ahora el groupby es todo sum/max — sin lambdas
agg = bb.groupby('SK_ID_BUREAU').agg(
    months_count = ('MONTHS_BALANCE', 'count'),
    months_span  = ('MONTHS_BALANCE', lambda x: x.max() - x.min()),
    n_mora       = ('is_mora',    'sum'),
    n_status_0   = ('is_0',       'sum'),
    n_status_C   = ('is_C',       'sum'),
    n_status_X   = ('is_X',       'sum'),
    max_severity = ('severity',   'max'),
).reset_index()

# Ratios y flags derivados
agg['pct_mora'] = agg['n_mora']     / agg['months_count']
agg['pct_X']    = agg['n_status_X'] / agg['months_count']
agg['has_mora'] = (agg['n_mora'] > 0).astype(int)

In [7]:
agg

,SK_ID_BUREAU,months_count,months_span,n_mora,n_status_0,n_status_C,n_status_X,max_severity,pct_mora,pct_X,has_mora
0,5001709,97,96,0,0,86,11,0,0.000000,0.113402,0
1,5001710,83,82,0,5,48,30,0,0.000000,0.361446,0
2,5001711,4,3,0,3,0,1,0,0.000000,0.250000,0
3,5001712,19,18,0,10,9,0,0,0.000000,0.000000,0
4,5001713,22,21,0,0,0,22,0,0.000000,1.000000,0
...,...,...,...,...,...,...,...,...,...,...,...
817390,6842884,48,47,0,9,20,19,0,0.000000,0.395833,0
817391,6842885,24,23,12,12,0,0,5,0.500000,0.000000,1
817392,6842886,33,32,0,8,25,0,0,0.000000,0.000000,0
817393,6842887,37,36,0,6,31,0,0,0.000000,0.000000,0
